# SmartPantry AI - Storage Guidance

## Notebook Overview

This notebook documents the development and refinement
of SmartPantry AI's source-based storage guidance system.

### Development History

| Stage | Description |
|---|---|
| Initial Role 3 prototype | Priority calculations and refrigerated leftovers guidance |
| October 24, 2026 | USDA FoodKeeper integration |
| Phase 5 - November 25-27, 2026 | Expanded storage metadata, source-specific validation, and application integration |

### Reading This Notebook

The original prototype and FoodKeeper integration sections
are preserved as historical development records.

Earlier implementation descriptions, test counts, and
integration statuses reflect their respective development
stages rather than the current application state.

The Phase 5 section demonstrates the current implementation
by importing functions directly from the SmartPantry
storage guidance module.

### Guiding Principle

**We do not guess dates we cannot source.**

Recommended use reminders are informational and do not
determine whether food is safe to consume.

In [35]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    project_root = Path("/content/smartpantry-ai")

    if not project_root.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "main",
                "https://github.com/SmartPantry-AI/smartpantry-ai.git",
                str(project_root),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(project_root / "requirements.txt"),
        ],
        check=True,
    )

else:
    project_root = Path.cwd().resolve()

    if not (project_root / "modules").is_dir():
        project_root = project_root.parent

assert (project_root / "modules" / "storage_guidance.py").is_file(), (
    "SmartPantry project folder not found."
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("SmartPantry project path configured.")
print("Project root:", project_root)

import os

os.chdir(project_root)

print("Working directory:", Path.cwd())

Environment: Local
SmartPantry project path configured.
Project root: C:\Users\Samir\GitHub\smartpantry-ai
Working directory: C:\Users\Samir\GitHub\smartpantry-ai


In [34]:
from datetime import date

def calculate_use_priority(days_remaining):
    """Rank reminder urgency; this does not determine food safety."""
    if days_remaining is None:
        return "UNKNOWN"
    if days_remaining <= 2:
        return "HIGH"
    if days_remaining <= 5:
        return "MEDIUM"
    return "LOW"


# First check: no storage guidance is being estimated yet.
for days in [None, -1, 0, 2, 3, 5, 6]:
    print(f"Days remaining: {str(days):>4} → {calculate_use_priority(days)}")

Days remaining: None → UNKNOWN
Days remaining:   -1 → HIGH
Days remaining:    0 → HIGH
Days remaining:    2 → HIGH
Days remaining:    3 → MEDIUM
Days remaining:    5 → MEDIUM
Days remaining:    6 → LOW


In [2]:
def evaluate_reminder(item_id, reminder_date=None, *, as_of):
    """Use a supplied reminder date without estimating storage duration."""
    days_remaining = (
        (date.fromisoformat(reminder_date) - as_of).days
        if reminder_date is not None else None
    )

    if days_remaining is None:
        status = "MISSING_DATE"
    elif days_remaining < 0:
        status = "PAST_REMINDER"
    elif days_remaining == 0:
        status = "TODAY"
    else:
        status = "UPCOMING"

    return {
        "item_id": item_id,
        "reminder_date": reminder_date,
        "use_priority": calculate_use_priority(days_remaining),
        "days_remaining": days_remaining,
        "reminder_status": status,
    }


# Fixed dates make this example reproducible.
print(evaluate_reminder(
    "demo-001", "2026-10-15", as_of=date(2026, 10, 13)
))
print(evaluate_reminder(
    "demo-002", as_of=date(2026, 10, 13)
))

{'item_id': 'demo-001', 'reminder_date': '2026-10-15', 'use_priority': 'HIGH', 'days_remaining': 2, 'reminder_status': 'UPCOMING'}
{'item_id': 'demo-002', 'reminder_date': None, 'use_priority': 'UNKNOWN', 'days_remaining': None, 'reminder_status': 'MISSING_DATE'}


In [3]:
expected_priorities = {
    None: "UNKNOWN",
    -1: "HIGH",
    0: "HIGH",
    2: "HIGH",
    3: "MEDIUM",
    5: "MEDIUM",
    6: "LOW",
}

for days, expected in expected_priorities.items():
    assert calculate_use_priority(days) == expected, (
        f"Priority failed for {days} days"
    )

reference_date = date(2026, 10, 13)

date_cases = [
    ("2026-10-12", -1, "PAST_REMINDER"),
    ("2026-10-13", 0, "TODAY"),
    ("2026-10-15", 2, "UPCOMING"),
    (None, None, "MISSING_DATE"),
]

for reminder, expected_days, expected_status in date_cases:
    result = evaluate_reminder(
        "test-item", reminder, as_of=reference_date
    )
    assert result["days_remaining"] == expected_days
    assert result["reminder_status"] == expected_status
    assert result["reminder_date"] == reminder

print("Passed: 7 priority cases and 4 reminder-date cases.")


Passed: 7 priority cases and 4 reminder-date cases.


In [5]:
from pathlib import Path
import csv
csv_path = Path("data/processed/storage_rules_clean.csv")
if not csv_path.exists():
    csv_path = Path("../data/processed/storage_rules_clean.csv")
with csv_path.open(newline="", encoding="utf-8-sig") as f:
    storage_rules = list(csv.DictReader(f))
supported_rule = next(row for row in storage_rules if row["source_record_id"] == "FS-COLD-020")
print("Records loaded:", len(storage_rules))


Records loaded: 20


In [6]:
supported_rule = next(
    row for row in storage_rules
    if row["source_record_id"] == "FS-COLD-020"
)

for field in [
    "source_record_id",
    "food_group",
    "food_type",
    "storage_location",
    "refrigerator_min_days",
    "refrigerator_max_days",
    "duration_condition",
    "source_url",
]:
    print(f"{field}: {supported_rule[field]}")


source_record_id: FS-COLD-020
food_group: Leftovers
food_type: Cooked meat or poultry
storage_location: REFRIGERATOR
refrigerator_min_days: 3
refrigerator_max_days: 4
duration_condition: NONE
source_url: https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts


## First supported storage rule
- Rule ID: FS-COLD-020
- Food: cooked meat or poultry leftovers
- Storage: refrigerator
- Required date: prepared date
- Dataset guidance: 3–4 days
- Prototype choice: use day 3 as the estimated use-soon reminder.
- Preserve the prepared date and retain the full source range.
- Missing context or unsupported food/storage returns UNKNOWN.

This is a reminder calculation, not a determination of food safety.
The day-3 choice is a prototype policy pending team review.

In [7]:
supported_rule = next(
    row for row in storage_rules
    if row["source_record_id"] == "FS-COLD-020"
)

print("Rule:", supported_rule["source_record_id"])
print("Minimum days:", supported_rule["refrigerator_min_days"])
print("Maximum days:", supported_rule["refrigerator_max_days"])

Rule: FS-COLD-020
Minimum days: 3
Maximum days: 4


In [8]:
from datetime import timedelta

def evaluate_leftovers(item, *, as_of):
    """Apply FS-COLD-020 only to explicitly confirmed matching items."""
    result = {
        "item_id": item["item_id"],
        "use_priority": "UNKNOWN",
        "days_remaining": None,
        "recommended_use_date": None,
        "prepared_date": item.get("prepared_date"),
        "reminder_status": "UNKNOWN",
        "source": None,
        "reason": "",
    }

    if (
        item.get("food_type") != "Cooked meat or poultry"
        or item.get("storage_location") != "REFRIGERATOR"
    ):
        result["reason"] = "Food or storage is unsupported by this rule."
        return result

    if not item.get("prepared_date"):
        result["reason"] = "Prepared date is required."
        return result

    try:
        prepared = date.fromisoformat(item["prepared_date"])
    except (TypeError, ValueError):
        result["reason"] = "Prepared date must be a valid YYYY-MM-DD date."
        return result

    if prepared > as_of:
        result["reason"] = "Prepared date is in the future; confirm it."
        return result

    reminder = prepared + timedelta(
        days=int(supported_rule["refrigerator_min_days"])
    )
    countdown = evaluate_reminder(
        item["item_id"], reminder.isoformat(), as_of=as_of
    )

    result.update({
        "use_priority": countdown["use_priority"],
        "days_remaining": countdown["days_remaining"],
        "recommended_use_date": reminder.isoformat(),
        "reminder_status": countdown["reminder_status"],
        "source": supported_rule["source_url"],
        "reason": "Estimated day-3 reminder using FS-COLD-020.",
    })
    return result


demo_item = {
    "item_id": "leftovers-001",
    "food_type": "Cooked meat or poultry",
    "storage_location": "REFRIGERATOR",
    "prepared_date": "2026-10-12",
}

print(evaluate_leftovers(demo_item, as_of=date(2026, 10, 13)))

{'item_id': 'leftovers-001', 'use_priority': 'HIGH', 'days_remaining': 2, 'recommended_use_date': '2026-10-15', 'prepared_date': '2026-10-12', 'reminder_status': 'UPCOMING', 'source': 'https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts', 'reason': 'Estimated day-3 reminder using FS-COLD-020.'}


In [9]:
reference_date = date(2026, 10, 13)

cases = [
    ("Valid leftovers", {}, "HIGH", 2),
    ("Missing prepared date", {"prepared_date": None}, "UNKNOWN", None),
    ("Invalid date", {"prepared_date": "not-a-date"}, "UNKNOWN", None),
    ("Future date", {"prepared_date": "2026-10-14"}, "UNKNOWN", None),
    ("Unsupported storage", {"storage_location": "FREEZER"}, "UNKNOWN", None),
    ("Unsupported food", {"food_type": "Spinach"}, "UNKNOWN", None),
    ("Reminder today", {"prepared_date": "2026-10-10"}, "HIGH", 0),
    ("Past reminder", {"prepared_date": "2026-10-09"}, "HIGH", -1),
]

for label, changes, expected_priority, expected_days in cases:
    item = {**demo_item, **changes}
    original = item.copy()
    result = evaluate_leftovers(item, as_of=reference_date)

    assert result["use_priority"] == expected_priority, label
    assert result["days_remaining"] == expected_days, label
    assert item == original, f"{label}: input was changed"
    assert result["prepared_date"] == item.get("prepared_date"), label

    print(f"PASS: {label}")

print(f"\nPassed: {len(cases)} leftovers cases.")


PASS: Valid leftovers
PASS: Missing prepared date
PASS: Invalid date
PASS: Future date
PASS: Unsupported storage
PASS: Unsupported food
PASS: Reminder today
PASS: Past reminder

Passed: 8 leftovers cases.


## Evaluation results — initial prototype
Reference date: 2026-10-13

- Priority cases: 7/7 passed.
- Reminder-date cases: 4/4 passed.
- Leftovers cases: 8/8 passed.
- Total: 19/19 passed.

Checks covered priority boundaries, date countdowns, missing/invalid/future
prepared dates, unsupported food/storage, and preservation of inputs.

Limitations:
- Automatic storage guidance currently supports only refrigerated
  cooked meat or poultry leftovers.
- Non-meat guidance, recall integration, Role 5 ranking integration,
  and user-facing notifications are not yet implemented or tested.
- Passing these tests does not establish food safety.

## Pending team decision: non-meat demo foods
The team has not yet confirmed the demo foods.

Role 3 needs the exact food types, storage locations, and relevant
conditions (such as opened/unopened or whole/cut) before selecting
and verifying additional storage rules.

In [10]:
import json

sample_items = [
    {**demo_item, "item_id": "sample-001"},
    {
        **demo_item,
        "item_id": "sample-002",
        "prepared_date": None,
    },
    {
        **demo_item,
        "item_id": "sample-003",
        "prepared_date": "2026-10-09",
    },
]

sample_outputs = [
    evaluate_leftovers(item, as_of=date(2026, 10, 13))
    for item in sample_items
]

print(json.dumps(sample_outputs, indent=2))


[
  {
    "item_id": "sample-001",
    "use_priority": "HIGH",
    "days_remaining": 2,
    "recommended_use_date": "2026-10-15",
    "prepared_date": "2026-10-12",
    "reminder_status": "UPCOMING",
    "source": "https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts",
    "reason": "Estimated day-3 reminder using FS-COLD-020."
  },
  {
    "item_id": "sample-002",
    "use_priority": "UNKNOWN",
    "days_remaining": null,
    "recommended_use_date": null,
    "prepared_date": null,
    "reminder_status": "UNKNOWN",
    "source": null,
    "reason": "Prepared date is required."
  },
  {
    "item_id": "sample-003",
    "use_priority": "HIGH",
    "days_remaining": -1,
    "recommended_use_date": "2026-10-12",
    "prepared_date": "2026-10-09",
    "reminder_status": "PAST_REMINDER",
    "source": "https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts",
    "reason": "Estimated day-3 reminder using FS-COLD-020."
  }
]


In [11]:
print(repr(supported_rule["source_url"]))

'https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts'


In [12]:
from urllib.parse import urlsplit
def normalize_source_url(value):
    url = value.strip()
    parts = urlsplit(url)
    if parts.scheme != 'https' or not parts.netloc: raise ValueError('Invalid URL')
    return url
clean_url = normalize_source_url(supported_rule["source_url"])
assert clean_url == "https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts"
print("PASS: source is a plain URL.")

PASS: source is a plain URL.


## Role 3 Handoff to Samira — Initial Prototype

Owner: Mahdi Mohammadkhani
Status: Working prototype; not yet integration-ready.

### Implemented
- Loads the cleaned 20-record storage dataset.
- Applies rule FS-COLD-020 to refrigerated cooked meat/poultry leftovers.
- Calculates an estimated day-3 reminder from the prepared date.
- Preserves the original prepared date.
- Returns HIGH, MEDIUM, LOW, or UNKNOWN priority.
- Includes 19 passing calculation/test cases and 3 sample outputs.

### Inputs for evaluate_leftovers()
- item_id
- food_type: "Cooked meat or poultry"
- storage_location: "REFRIGERATOR"
- prepared_date: YYYY-MM-DD
- as_of: explicit Python date used for calculation

### Outputs
- item_id
- use_priority
- days_remaining
- recommended_use_date
- prepared_date
- reminder_status
- source
- reason

### Integration boundaries
Role 5 consumes use_priority rather than recreating storage rules.
Recall eligibility is handled by Role 4 and the integration pipeline.
HIGH priority does not establish food safety or recipe eligibility.

### Pending
- Confirm shared field names and demo foods.
- Verify and add selected non-meat guidance.
- Implement user-facing reminder messages.
- Package the function in the repository module.
- Test the handoff with Role 5 ranking.

### Questions for Samira
1. Can your ranking module consume use_priority, or does it currently
   require expiration_priority?
2. Which non-meat foods and storage conditions will the demo use?
3. How should UNKNOWN and PAST_REMINDER outputs be handled?

### Response to Question 1: Role 5 Ranking Compatibility

**Status: Identified integration requirement.**

Role 5 currently calculates recipe-ranking urgency using the pantry field `days_until_expiration`.

Role 3 instead returns `days_remaining`, `use_priority`, `recommended_use_date`, and `reminder_status`.

Neither `use_priority` nor `expiration_priority` is currently consumed by the Role 5 ranking function.

**Decision:** Preserve Role 3's source-backed reminder calculations and introduce an explicit mapping between Role 3 guidance and Role 5 recipe-ranking urgency.

This mapping must distinguish use-soon reminders from confirmed expiration dates and safely handle `UNKNOWN` values.

**Remaining work:** Implement and test the Role 3-to-Role 5 ranking handoff.

### Response to Question 2: Demo Foods and Storage Conditions

**Status: Proposed demo inputs; final team selection pending.**

The current Role 3 integration supports the following demonstration cases:

| Food | Storage condition | Required date | Expected result |
|---|---|---|---|
| Spinach | REFRIGERATOR | Purchase date | FoodKeeper use-soon reminder |
| Regular tomatoes | To be confirmed | N/A | UNKNOWN |
| Milk | REFRIGERATOR | Package use-by date for future support | UNKNOWN under current purchase-date evaluator |
| Cooked meat/poultry | REFRIGERATOR | Prepared date | Original Role 3 reminder |

**Spinach example:** With a purchase date of October 24, 2026, the FoodKeeper 3–7-day refrigerated reference range produces a use-soon reminder for October 27, 2026.

**Regular tomatoes:** The current FoodKeeper matching logic does not assign cherry-tomato guidance to regular tomatoes.

**Milk:** The existing FoodKeeper record refers to the package use-by date. The current implementation does not calculate a reminder from that field.

**Demo decision:** Use spinach to demonstrate newly supported FoodKeeper guidance, and use regular tomatoes to demonstrate why unsupported storage dates remain UNKNOWN.

The final demo food selection and storage inputs must be confirmed with the team.

### Response to Question 3: UNKNOWN and PAST_REMINDER Handling

**Status: Proposed integration behavior; implementation pending.**

#### UNKNOWN

`UNKNOWN` indicates that Role 3 cannot calculate a reliable use-soon reminder from the available information.

Expected behavior:

- Preserve `UNKNOWN` rather than inventing an expiration date.
- Display the explanation provided by Role 3's `reason` field.
- Assign no urgency-based recipe-ranking points without supported guidance.
- Allow the ingredient to count toward pantry availability independently of its storage priority.
- Request missing information, such as a purchase date, when appropriate.

#### PAST_REMINDER

`PAST_REMINDER` indicates that a calculated reminder date has already passed.

Expected behavior:

- Display that the recommended use-soon date has passed.
- Preserve the original date and source.
- Do not automatically label the food expired, unsafe, or recalled.
- Do not automatically prioritize the ingredient for cooking solely because its reminder is overdue.
- Require separate safety or eligibility decisions before recommending its use.

#### Role 5 Integration

Role 5 currently expects numeric `days_until_expiration` values.

Role 3 provides `days_remaining`, `use_priority`, and `reminder_status`.

An explicit integration mapping is required to handle these values safely, particularly when `days_remaining` is `None` or the reminder date has passed.

**Decision:** Treat storage reminders as prioritization guidance, not proof of food safety. Keep recipe ranking separate from food-safety and recall eligibility.

**Remaining work:** Implement and test these behaviors in the application workflow.



## Role 3 Handoff: Updated Integration Status

**Updated:** October 24, 2026

### Completed

- Preserved the original cooked-meat and poultry storage evaluator.
- Added USDA FoodKeeper data loading and validation.
- Implemented explicit FoodKeeper food matching.
- Added source-backed refrigerated spinach guidance.
- Implemented safeguards for unsupported foods and missing dates.
- Added four automated FoodKeeper tests.
- Confirmed all seven Role 3 automated tests pass.
- Documented the Role 3-to-Role 5 field mismatch.

### Remaining Integration Work

1. Confirm final demo foods and shared input fields.
2. Connect Role 3 outputs to Role 5 recipe ranking.
3. Implement user-facing reminder messages.
4. Verify UNKNOWN and PAST_REMINDER handling across the workflow.
5. Run end-to-end integration tests with the team's demo inputs.

### Integration Decision

Role 3 remains responsible for source-backed storage guidance.

Role 5 remains responsible for recipe ranking and must not
reinterpret storage reminders as verified expiration dates.

Role 4 and the integration pipeline remain responsible for
recall-related eligibility decisions.

### Handoff Status

**Role 3 FoodKeeper implementation: Unit-tested and documented.**

**Complete application integration: Pending.**

---

# Role 3: USDA FoodKeeper Integration

**Integration date:** October 24, 2026  
**Branch:** `role-3-foodkeeper-integration`  
**Module:** `modules/storage_guidance.py`  
**Tests:** `tests/test_storage.py`

## Integration Objective

Extend the original Role 3 storage guidance system to support
additional foods using the cleaned USDA FoodKeeper dataset.

The original implementation supported cooked meat and poultry
refrigeration reminders. The integration adds a separate FoodKeeper
evaluation pathway without modifying that original functionality.

## Why This Change Was Necessary

SmartPantry can identify foods through image recognition and
barcode scanning, but identifying an item does not automatically
provide reliable storage guidance.

Previously, foods such as spinach returned UNKNOWN because
the original storage rules did not cover them.

The FoodKeeper integration introduces source-backed storage
guidance while preserving UNKNOWN when the available data
does not support a calculation.

**Design principle:** We do not guess dates we cannot source.

## FoodKeeper Dataset and Source Validation

**Source:** USDA FSIS FoodKeeper Data (EN)  
**Source URL:** https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls  
**Processed dataset:** `data/processed/foodkeeper_produce_dairy_shelf_life_clean.csv`

### Dataset Preparation

The cleaned dataset contains 17 records covering selected produce and dairy products.

Each record includes:

- `food_name`: FoodKeeper food description.
- `foodkeeper_category`: Food category.
- `fridge_days_min`: Minimum refrigerated storage duration.
- `fridge_days_max`: Maximum refrigerated storage duration.
- `basis`: Reference point for interpreting the duration.
- `note`: Additional storage guidance.
- `source`: Original data source.

### Food Matching and Date Calculations

The integration uses explicit food-name matching to avoid assigning incorrect guidance.

**Spinach:** Matches the FoodKeeper entry `lettuce, leaf / spinach`, which provides a refrigerated reference range of 3–7 days from the purchase date.

**Regular tomatoes:** Remain `UNKNOWN` because the available cherry-tomato entry does not establish guidance for regular tomatoes.

**Milk:** Remains `UNKNOWN` for purchase-date calculations because its record references the package use-by date rather than a numeric duration from purchase.

### Storage Guidance Rules

A FoodKeeper-based reminder is calculated only when:

1. The food matches a supported record.
2. The storage location is `REFRIGERATOR`.
3. The record explicitly supports calculations from the purchase date.
4. A valid purchase date is available.
5. The dataset provides a valid numeric storage range.

When these requirements are satisfied, the earliest supported day is used to calculate a conservative use-soon reminder.

**Important:** FoodKeeper storage ranges are reference guidance, not guaranteed food-safety expiration dates. Unsupported or incomplete information produces `UNKNOWN` rather than an invented date.

## FoodKeeper Integration: Implementation

### 1. `load_foodkeeper_rules()`

**Purpose:** Load the cleaned FoodKeeper dataset.

- Reads the CSV using `csv.DictReader`.
- Validates that the required columns exist.
- Checks that the dataset contains records.
- Returns the records for food matching and evaluation.

**Validation result:** 17 records loaded successfully.

### 2. `find_foodkeeper_rule()`

**Purpose:** Match a recognized food to an appropriate FoodKeeper record.

The function normalizes food names and uses explicit mappings
rather than assuming that similar foods share storage guidance.

Examples:

| Recognized food | FoodKeeper match |
|---|---|
| Spinach | lettuce, leaf / spinach |
| Cherry tomatoes | cherry tomatoes |
| Regular tomato | No match |
| Milk | milk |

A successful name match does not automatically mean that a
storage date can be calculated.

### 3. `evaluate_foodkeeper()`

**Purpose:** Evaluate whether reliable storage guidance can be calculated.

The function:

1. Identifies the food using `find_foodkeeper_rule()`.
2. Verifies refrigerated storage.
3. Checks that the FoodKeeper basis supports purchase-date calculations.
4. Validates the purchase date.
5. Validates the numeric storage range.
6. Calculates the earliest recommended use-soon date.
7. Uses `evaluate_reminder()` to determine priority and reminder status.
8. Returns the guidance, source, and explanation.

When a required condition is not satisfied, the result remains
`UNKNOWN` with an explanatory reason.

### Preservation of Original Functionality

The original `evaluate_leftovers()` function remains unchanged.

FoodKeeper evaluation is implemented as a separate pathway so
that the existing cooked-meat and poultry rules continue working.

**Integration consideration:** The calling workflow must select
the appropriate evaluation pathway and supply the required fields.

## FoodKeeper Validation and Test Results

### Example: Spinach Storage Guidance

**Test input:**

- Item ID: `spinach-001`
- Food type: `Spinach`
- Storage location: `REFRIGERATOR`
- Purchase date: `2026-10-24`
- Evaluation date: `2026-10-25`

**Verified output:**

| Output field | Result |
|---|---|
| Use priority | HIGH |
| Days remaining | 2 |
| Recommended use-soon date | 2026-10-27 |
| Reminder status | UPCOMING |
| Storage range | 3–7 days |
| Storage range end date | 2026-10-31 |
| Source | USDA FSIS FoodKeeper Data (EN) |

The use-soon date is calculated using the earliest supported
storage day: October 24 + 3 days = October 27.

This is a reminder for prioritizing food use, not a guaranteed
food-safety expiration date.

### Automated Regression Testing

**Test file:** `tests/test_storage.py`

**Command:**

`python -m unittest tests.test_storage -v`

| Test | Result |
|---|---|
| test_dataset | PASS |
| test_leftovers | PASS |
| test_unsupported | PASS |
| test_foodkeeper_spinach | PASS |
| test_foodkeeper_missing_purchase_date | PASS |
| test_foodkeeper_regular_tomato | PASS |
| test_foodkeeper_milk | PASS |

**Final result: 7 tests passed.**

The original three tests continue to pass, and four additional
tests verify the FoodKeeper integration.

### Additional Manual Validation

Manual checks also confirmed:

- The cleaned FoodKeeper dataset loads 17 records.
- Spinach without a purchase date returns UNKNOWN.
- Regular tomatoes return UNKNOWN rather than borrowing cherry-tomato guidance.
- Milk returns UNKNOWN because its FoodKeeper basis does not support purchase-date calculations.

### Test Evidence

See `role3_storage_test_results.txt` in the test results folder
for the preserved original pytest baseline and the integration
test summary.

## Role 3: Integration With SmartPantry

### Role-to-Workflow Connections

Role 3 provides storage guidance after SmartPantry identifies a food item.

**Role 2: Barcode Recognition**
- Identifies packaged foods using barcode data.
- Provides available product information.
- Product names must be mapped to the food names expected by Role 3.
- A successful barcode lookup does not guarantee that storage guidance is available.

**Role 5: Image Recognition and AI Assistant**
- Identifies foods from images.
- Can provide recognized food names to the storage guidance workflow.
- Spinach is supported by the new FoodKeeper evaluation pathway.
- Regular tomatoes remain UNKNOWN unless a suitable source-backed rule is added.

**Role 3: Storage Guidance**
- Receives the food name and relevant storage information.
- Determines whether a supported storage rule exists.
- Calculates a use-soon reminder only when sufficient information is available.
- Returns UNKNOWN when the evidence does not support a calculation.

### Required Inputs

For FoodKeeper purchase-date calculations, the workflow needs:

- `item_id`
- `food_type`
- `storage_location`
- `purchase_date`

The evaluation also requires an `as_of` date.

A recognized food name alone is not sufficient to calculate a storage reminder.

### Integration Requirements

Before the complete workflow can be considered verified:

1. Confirm that Roles 2 and 5 provide compatible food-name fields.
2. Confirm how storage location and purchase date are collected.
3. Connect the appropriate storage evaluator to the application workflow.
4. Verify that supported foods produce source-backed reminders.
5. Verify that unsupported foods retain UNKNOWN.
6. Run end-to-end tests using the actual demo inputs.

### Demo Behavior

**Spinach:** Can receive a FoodKeeper-based use-soon reminder when a valid purchase date and refrigerated storage are provided.

**Regular tomatoes:** Return UNKNOWN because the current supported records do not establish an appropriate purchase-date calculation.

**Milk:** Returns UNKNOWN for purchase-date calculations because its FoodKeeper record references package use-by guidance.

**Cooked chicken:** Continues to use the original Role 3 cooked-meat and poultry rules.

### Current Status

- FoodKeeper data loading: VERIFIED
- Food matching: VERIFIED
- Spinach reminder calculation: VERIFIED
- Unsupported-food safeguards: VERIFIED
- Original Role 3 regression tests: PASS
- Role 3 automated tests: 7 PASSED
- Complete SmartPantry end-to-end integration: NOT YET VERIFIED

**Design principle:** We do not guess dates we cannot source.

# Phase 5 - Storage Guidance Refinement

**Development Period:** November 25-27, 2026

## 1. Environment Setup

This section validates the current SmartPantry AI storage
guidance implementation using functions imported directly
from the project modules.

All demonstrations use controlled test inputs and fixed
reference dates to ensure reproducible results.

No live API calls or database modifications are required.

The historical prototype functions defined earlier in
this notebook are not used for Phase 5 validation.

In [36]:
from datetime import date
from modules import storage_guidance as storage

print("Phase 5 storage module imported successfully.")
print("Project root:", project_root)

Phase 5 storage module imported successfully.
Project root: C:\Users\Samir\GitHub\smartpantry-ai


## 2. Expanded FoodKeeper Dataset

The Phase 5 FoodKeeper dataset contains 17 selected
storage guidance records with source identifiers,
storage conditions, calculation bases, and storage notes.

Only records with supported calculation bases and
satisfied metadata requirements are eligible for
recommended use date calculations.

This section verifies the dataset structure and
distribution of calculation bases.

In [14]:
import csv
from collections import Counter

foodkeeper_path = (
    project_root
    / "data"
    / "processed"
    / "foodkeeper_produce_dairy_shelf_life_clean.csv"
)

with foodkeeper_path.open(
    encoding="utf-8-sig", newline=""
) as file:
    foodkeeper_records = list(csv.DictReader(file))

basis_counts = Counter(
    record["basis"] for record in foodkeeper_records
)

print("Total FoodKeeper records:", len(foodkeeper_records))

print("\nCalculation bases:")

for basis, count in basis_counts.items():
    print(f"{basis}: {count}")

print("\nDataset columns:")

for column in foodkeeper_records[0]:
    print(column)

Total FoodKeeper records: 17

Calculation bases:
package use-by date: 1
from date of purchase: 10
after opening: 2
refrigerated: 4

Dataset columns:
food_name
foodkeeper_category
fridge_days_min
fridge_days_max
basis
note
source
source_record_id
required_food_classification
required_preparation_state
required_package_state


### Dataset Structure Validation

The following assertions verify the expected dataset
size, required columns, unique source identifiers,
and distribution of calculation bases.

These checks establish the expected Phase 5 dataset
baseline and detect unexpected structural changes.

In [15]:
expected_columns = {
    "food_name",
    "foodkeeper_category",
    "fridge_days_min",
    "fridge_days_max",
    "basis",
    "note",
    "source",
    "source_record_id",
    "required_food_classification",
    "required_preparation_state",
    "required_package_state",
}

expected_basis_counts = {
    "package use-by date": 1,
    "from date of purchase": 10,
    "after opening": 2,
    "refrigerated": 4,
}

source_ids = [
    record["source_record_id"]
    for record in foodkeeper_records
]

assert len(foodkeeper_records) == 17
assert set(foodkeeper_records[0]) == expected_columns
assert basis_counts == expected_basis_counts
assert len(set(source_ids)) == len(source_ids)
assert all(source_id.isdigit() for source_id in source_ids)

print("PASS: Dataset contains 17 records.")
print("PASS: All 11 expected columns are present.")
print("PASS: Calculation basis distribution is correct.")
print("PASS: Source record identifiers are unique.")
print("PASS: All source record identifiers are numeric.")
print("\nDataset validation: 5/5 checks passed.")

PASS: Dataset contains 17 records.
PASS: All 11 expected columns are present.
PASS: Calculation basis distribution is correct.
PASS: Source record identifiers are unique.
PASS: All source record identifiers are numeric.

Dataset validation: 5/5 checks passed.


## 3. Purchase-Based Reminder Calculation

This demonstration validates a FoodKeeper storage rule
that calculates a recommended use reminder from the
purchase date.

### Test Scenario

- Ingredient: Apple
- Purchase date: November 24, 2026
- Reference date: November 27, 2026
- Storage location: REFRIGERATOR
- Food classification: REGULAR
- Preparation state: WHOLE
- Supported storage duration: 28-42 days

The earliest supported day is used to calculate the
recommended use reminder.

Expected reminder: December 22, 2026.

In [16]:
reference_date = date(2026, 11, 27)

apple_item = {
    "item_id": "phase5-apple-001",
    "food_type": "apple",
    "storage_location": "REFRIGERATOR",
    "purchase_date": "2026-11-24",
    "food_classification": "REGULAR",
    "preparation_state": "WHOLE",
    "package_state": None,
    "prepared_date": None,
    "opened_date": None,
    "thawed_date": None,
}

apple_result = storage.evaluate_foodkeeper(
    apple_item,
    as_of=reference_date,
    foodkeeper_rules=foodkeeper_records,
)

for key, value in apple_result.items():
    print(f"{key}: {value}")

item_id: phase5-apple-001
use_priority: LOW
days_remaining: 25
recommended_use_date: 2026-12-22
reminder_status: UPCOMING
source: USDA FSIS FoodKeeper Data (EN), https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls
reason: Use-soon reminder based on FoodKeeper's earliest supported day.
storage_range_days: (28, 42)
storage_range_end_date: 2027-01-05
calculation_basis: Purchase date


In [17]:
assert str(apple_result["recommended_use_date"]) == "2026-12-22"
assert str(apple_result["storage_range_end_date"]) == "2027-01-05"
assert apple_result["days_remaining"] == 25
assert apple_result["use_priority"] == "LOW"
assert apple_result["reminder_status"] == "UPCOMING"
assert apple_result["storage_range_days"] == (28, 42)
assert apple_result["calculation_basis"] == "Purchase date"
assert "FoodKeeper" in apple_result["source"]

print("PASS: Recommended use date.")
print("PASS: Storage range end date.")
print("PASS: Days remaining.")
print("PASS: Use priority.")
print("PASS: Reminder status.")
print("PASS: Storage duration range.")
print("PASS: Calculation basis.")
print("PASS: Source attribution.")

print("\nPurchase-based validation: 8/8 checks passed.")

PASS: Recommended use date.
PASS: Storage range end date.
PASS: Days remaining.
PASS: Use priority.
PASS: Reminder status.
PASS: Storage duration range.
PASS: Calculation basis.
PASS: Source attribution.

Purchase-based validation: 8/8 checks passed.


## 4. Opening-Based Reminder Calculation

This demonstration validates a FoodKeeper rule that
requires an opened package and a valid opening date.

### Test Scenario

- Ingredient: Cheddar cheese
- Purchase date: November 15, 2026
- Opening date: November 24, 2026
- Reference date: November 27, 2026
- Storage location: REFRIGERATOR
- Food classification: REGULAR
- Package state: OPENED
- Supported storage duration: 21-28 days

The recommended use reminder is calculated from the
opening date rather than the purchase date.

Expected reminder: December 15, 2026.

In [18]:
cheese_item = {
    "item_id": "phase5-cheese-001",
    "food_type": "cheddar cheese",
    "storage_location": "REFRIGERATOR",
    "purchase_date": "2026-11-15",
    "food_classification": "REGULAR",
    "preparation_state": None,
    "package_state": "OPENED",
    "prepared_date": None,
    "opened_date": "2026-11-24",
    "thawed_date": None,
}

cheese_result = storage.evaluate_foodkeeper(
    cheese_item,
    as_of=reference_date,
    foodkeeper_rules=foodkeeper_records,
)

for key, value in cheese_result.items():
    print(f"{key}: {value}")

item_id: phase5-cheese-001
use_priority: LOW
days_remaining: 18
recommended_use_date: 2026-12-15
reminder_status: UPCOMING
source: USDA FSIS FoodKeeper Data (EN), https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls
reason: Use-soon reminder based on FoodKeeper's earliest supported day.
storage_range_days: (21, 28)
storage_range_end_date: 2026-12-22
calculation_basis: Opening date


In [19]:
assert str(cheese_result["recommended_use_date"]) == "2026-12-15"
assert str(cheese_result["storage_range_end_date"]) == "2026-12-22"
assert cheese_result["days_remaining"] == 18
assert cheese_result["use_priority"] == "LOW"
assert cheese_result["reminder_status"] == "UPCOMING"
assert cheese_result["storage_range_days"] == (21, 28)
assert cheese_result["calculation_basis"] == "Opening date"
assert "FoodKeeper" in cheese_result["source"]

print("PASS: Recommended use date.")
print("PASS: Storage range end date.")
print("PASS: Days remaining.")
print("PASS: Use priority.")
print("PASS: Reminder status.")
print("PASS: Storage duration range.")
print("PASS: Calculation basis.")
print("PASS: Source attribution.")

print("\nOpening-based validation: 8/8 checks passed.")

PASS: Recommended use date.
PASS: Storage range end date.
PASS: Days remaining.
PASS: Use priority.
PASS: Reminder status.
PASS: Storage duration range.
PASS: Calculation basis.
PASS: Source attribution.

Opening-based validation: 8/8 checks passed.


## 5. Preparation-Based Reminder Calculation

This demonstration validates the FoodSafety.gov
storage rule FS-COLD-020 for cooked meat or poultry.

### Test Scenario

- Ingredient: Cooked meat or poultry
- Preparation date: November 24, 2026
- Reference date: November 27, 2026
- Storage location: REFRIGERATOR
- Food classification: COOKED
- Preparation state: COOKED
- Supported storage duration: 3-4 days

The recommended use reminder is calculated from the
preparation date using the earliest supported day.

Expected reminder: November 27, 2026.

The reminder is due on the reference date and does
not represent a food safety determination.

In [20]:
leftovers_item = {
    "item_id": "phase5-leftovers-001",
    "food_type": "cooked meat or poultry",
    "storage_location": "REFRIGERATOR",
    "purchase_date": None,
    "food_classification": "COOKED",
    "preparation_state": "COOKED",
    "package_state": None,
    "prepared_date": "2026-11-24",
    "opened_date": None,
    "thawed_date": None,
}

leftovers_result = storage.evaluate_leftovers(
    leftovers_item,
    as_of=reference_date,
)

for key, value in leftovers_result.items():
    print(f"{key}: {value}")

item_id: phase5-leftovers-001
use_priority: HIGH
days_remaining: 0
recommended_use_date: 2026-11-27
prepared_date: 2026-11-24
reminder_status: TODAY
source: https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts
reason: Estimated day-3 reminder using FS-COLD-020.
storage_range_days: (3, 4)
storage_range_end_date: 2026-11-28
calculation_basis: Preparation date


In [21]:
assert str(leftovers_result["recommended_use_date"]) == "2026-11-27"
assert str(leftovers_result["storage_range_end_date"]) == "2026-11-28"
assert leftovers_result["days_remaining"] == 0
assert leftovers_result["use_priority"] == "HIGH"
assert leftovers_result["reminder_status"] == "TODAY"
assert leftovers_result["storage_range_days"] == (3, 4)
assert leftovers_result["calculation_basis"] == "Preparation date"
assert "foodsafety.gov" in leftovers_result["source"]

print("PASS: Recommended use date.")
print("PASS: Storage range end date.")
print("PASS: Days remaining.")
print("PASS: Use priority.")
print("PASS: Reminder status.")
print("PASS: Storage duration range.")
print("PASS: Calculation basis.")
print("PASS: Source attribution.")

print("\nPreparation-based validation: 8/8 checks passed.")

PASS: Recommended use date.
PASS: Storage range end date.
PASS: Days remaining.
PASS: Use priority.
PASS: Reminder status.
PASS: Storage duration range.
PASS: Calculation basis.
PASS: Source attribution.

Preparation-based validation: 8/8 checks passed.


## 6. Unsupported Conditions and UNKNOWN Results

The storage guidance evaluator must withhold calculated
reminders when required information is missing or when
storage conditions are unsupported.

This section validates the following scenarios:

1. Missing food classification.
2. Missing purchase date.
3. Unsupported freezer storage.
4. Previously frozen food with a recorded thawing date.
5. Missing opening date.
6. Unopened package requiring an opening-based rule.
7. Missing preparation date.
8. Unsupported general refrigeration calculation.
9. Package use-by guidance without a supported calculation.

Each scenario is expected to return UNKNOWN without
generating a recommended use date.

An UNKNOWN result indicates insufficient information
or unsupported calculation conditions, not a
determination of food safety.

In [22]:
unsupported_cases = {
    "Missing classification": {
        **apple_item,
        "food_classification": None,
    },
    "Missing purchase date": {
        **apple_item,
        "purchase_date": None,
    },
    "Unsupported freezer storage": {
        **apple_item,
        "storage_location": "FREEZER",
    },
    "Previously frozen food": {
        **apple_item,
        "thawed_date": "2026-11-25",
    },
    "Missing opening date": {
        **cheese_item,
        "opened_date": None,
    },
    "Unopened package": {
        **cheese_item,
        "package_state": "UNOPENED",
        "opened_date": None,
    },
}

unsupported_results = {}

for scenario, item in unsupported_cases.items():
    result = storage.evaluate_foodkeeper(
        item,
        as_of=reference_date,
        foodkeeper_rules=foodkeeper_records,
    )

    unsupported_results[scenario] = result

    print(scenario)
    print("Priority:", result["use_priority"])
    print("Recommended use date:", result["recommended_use_date"])
    print("Reason:", result["reason"])
    print()

Missing classification
Priority: UNKNOWN
Recommended use date: None
Reason: Food classification is required for this FoodKeeper rule.

Missing purchase date
Priority: UNKNOWN
Recommended use date: None
Reason: Purchase date is required.

Unsupported freezer storage
Priority: UNKNOWN
Recommended use date: None
Reason: FoodKeeper rule requires refrigerated storage.

Previously frozen food
Priority: UNKNOWN
Recommended use date: None
Reason: Previously frozen food requires a supported thawing rule.

Missing opening date
Priority: UNKNOWN
Recommended use date: None
Reason: Opening date is required.

Unopened package
Priority: UNKNOWN
Recommended use date: None
Reason: Package state must be OPENED for this FoodKeeper rule.



In [23]:
expected_reasons = {
    "Missing classification":
        "Food classification is required",
    "Missing purchase date":
        "Purchase date is required",
    "Unsupported freezer storage":
        "requires refrigerated storage",
    "Previously frozen food":
        "requires a supported thawing rule",
    "Missing opening date":
        "Opening date is required",
    "Unopened package":
        "Package state must be OPENED",
}

for scenario, result in unsupported_results.items():
    assert result["use_priority"] == "UNKNOWN"
    assert result["recommended_use_date"] is None
    assert result["days_remaining"] is None
    assert result["calculation_basis"] is None
    assert expected_reasons[scenario] in result["reason"]

    print(f"PASS: {scenario} (5/5 checks)")

print("\nUnsupported conditions validation: 30/30 checks passed.")

PASS: Missing classification (5/5 checks)
PASS: Missing purchase date (5/5 checks)
PASS: Unsupported freezer storage (5/5 checks)
PASS: Previously frozen food (5/5 checks)
PASS: Missing opening date (5/5 checks)
PASS: Unopened package (5/5 checks)

Unsupported conditions validation: 30/30 checks passed.


### Additional Unsupported Calculation Scenarios

The presence of a matching storage rule does not
automatically establish calculation eligibility.

The following scenarios verify that SmartPantry
withholds reminders when:

- Cooked leftovers have no preparation date.
- A FoodKeeper rule provides a general refrigeration
  duration without a supported calculation basis.
- A FoodKeeper rule refers to a package use-by date
  rather than a numeric storage duration.

The evaluator must preserve UNKNOWN results rather
than substitute an unsupported calculation basis.

In [24]:
missing_preparation_item = {
    **leftovers_item,
    "prepared_date": None,
}

banana_item = {
    **apple_item,
    "item_id": "phase5-banana-001",
    "food_type": "banana",
}

milk_item = {
    **apple_item,
    "item_id": "phase5-milk-001",
    "food_type": "milk",
    "preparation_state": None,
}

additional_unsupported_results = {
    "Missing preparation date": storage.evaluate_leftovers(
        missing_preparation_item,
        as_of=reference_date,
    ),
    "General refrigeration basis": storage.evaluate_foodkeeper(
        banana_item,
        as_of=reference_date,
        foodkeeper_rules=foodkeeper_records,
    ),
    "Package use-by basis": storage.evaluate_foodkeeper(
        milk_item,
        as_of=reference_date,
        foodkeeper_rules=foodkeeper_records,
    ),
}

for scenario, result in additional_unsupported_results.items():
    print(scenario)
    print("Priority:", result["use_priority"])
    print("Recommended use date:", result["recommended_use_date"])
    print("Reason:", result["reason"])
    print()

Missing preparation date
Priority: UNKNOWN
Recommended use date: None
Reason: Prepared date is required.

General refrigeration basis
Priority: UNKNOWN
Recommended use date: None
Reason: FoodKeeper rule does not support date calculations.

Package use-by basis
Priority: UNKNOWN
Recommended use date: None
Reason: FoodKeeper rule does not support date calculations.



In [25]:
additional_expected_reasons = {
    "Missing preparation date":
        "Prepared date is required",
    "General refrigeration basis":
        "does not support date calculations",
    "Package use-by basis":
        "does not support date calculations",
}

for scenario, result in additional_unsupported_results.items():
    assert result["use_priority"] == "UNKNOWN"
    assert result["recommended_use_date"] is None
    assert result["days_remaining"] is None
    assert result["calculation_basis"] is None
    assert additional_expected_reasons[scenario] in result["reason"]

    print(f"PASS: {scenario} (5/5 checks)")

print("\nAdditional unsupported conditions: 15/15 checks passed.")

PASS: Missing preparation date (5/5 checks)
PASS: General refrigeration basis (5/5 checks)
PASS: Package use-by basis (5/5 checks)

Additional unsupported conditions: 15/15 checks passed.


## 7. Oregon State University Informational Guidance

Oregon State University provides supplemental produce
storage guidance within SmartPantry AI.

Unlike supported FoodKeeper and FoodSafety.gov rules,
OSU records are currently used for informational
purposes only.

### Test Scenario

- Ingredient: Asparagus
- Storage location: REFRIGERATOR
- Expected storage duration: 3-4 days
- Expected storage temperature: 35-40 degrees Fahrenheit

The source-supported duration and temperature are
displayed as informational guidance.

No recommended use date is calculated from the OSU
storage duration.

In [26]:
osu_result = storage.find_osu_rule("asparagus")

assert osu_result is not None, (
    "Expected OSU asparagus rule was not found."
)

for key, value in osu_result.items():
    print(f"{key}: {value}")

source_record_id: OSU-PNW612-001
source_organization: Oregon State University Extension / Pacific Northwest Extension
source_title: Storing Food for Safety and Quality (PNW 612)
source_table: Table 3: Approximate Storage Times for Best Quality Fresh Fruits and Vegetables
source_page: 18
food_type_raw: Asparagus
food_type: Asparagus
storage_location: REFRIGERATOR
storage_temperature_raw: 35-40 F
duration_raw: 3-4 days
refrigerator_min_days: 3
refrigerator_max_days: 4
duration_basis: NOT_SPECIFIED_IN_TABLE
requires_condition_review: True
prototype_supported: False
source_notes: 
source_url: https://extension.oregonstate.edu/sites/default/files/documents/pnw612.pdf
extraction_method: Manual transcription from official publication
extracted_date: 2026-11-14


In [27]:
assert osu_result["source_record_id"] == "OSU-PNW612-001"
assert osu_result["food_type"].lower() == "asparagus"
assert osu_result["storage_location"] == "REFRIGERATOR"
assert osu_result["storage_temperature_raw"] == "35-40 F"
assert osu_result["duration_raw"] == "3-4 days"
assert int(osu_result["refrigerator_min_days"]) == 3
assert int(osu_result["refrigerator_max_days"]) == 4
assert osu_result["duration_basis"] == "NOT_SPECIFIED_IN_TABLE"
assert str(osu_result["requires_condition_review"]).lower() == "true"
assert str(osu_result["prototype_supported"]).lower() == "false"
assert "oregonstate.edu" in osu_result["source_url"]

print("PASS: Source record identifier.")
print("PASS: Ingredient identification.")
print("PASS: Storage location.")
print("PASS: Storage temperature.")
print("PASS: Storage duration.")
print("PASS: Minimum storage duration.")
print("PASS: Maximum storage duration.")
print("PASS: Unspecified calculation basis.")
print("PASS: Condition review requirement.")
print("PASS: Calculation eligibility disabled.")
print("PASS: Source attribution.")

print("\nOSU informational guidance: 11/11 checks passed.")

PASS: Source record identifier.
PASS: Ingredient identification.
PASS: Storage location.
PASS: Storage temperature.
PASS: Storage duration.
PASS: Minimum storage duration.
PASS: Maximum storage duration.
PASS: Unspecified calculation basis.
PASS: Condition review requirement.
PASS: Calculation eligibility disabled.
PASS: Source attribution.

OSU informational guidance: 11/11 checks passed.


## 8. Pantry Metadata Adaptation

The pantry metadata adapter transforms database
records into the structured inputs required by
the storage guidance evaluators.

Phase 5 expanded the adapter to preserve:

- Food classification.
- Preparation state.
- Package state.
- Opening date.
- Thawing date.

This section uses simulated Supabase records to
validate field mapping, date normalization, and
backward compatibility with older pantry records.

No live database connection is required.

In [28]:
simulated_pantry_record = {
    "id": 5001,
    "name": "cheddar cheese",
    "quantity": 1,
    "unit": "item",
    "storage_location": "REFRIGERATOR",
    "purchase_date": date(2026, 11, 15),
    "expiration_date": None,
    "food_classification": "REGULAR",
    "preparation_state": None,
    "package_state": "OPENED",
    "prepared_date": None,
    "opened_date": date(2026, 11, 24),
    "thawed_date": None,
}

adapted_item = storage.adapt_pantry_item(
    simulated_pantry_record
)

for key, value in adapted_item.items():
    print(f"{key}: {value}")

item_id: 5001
food_type: cheddar cheese
storage_location: REFRIGERATOR
purchase_date: 2026-11-15
prepared_date: None
food_classification: REGULAR
preparation_state: None
package_state: OPENED
opened_date: 2026-11-24
thawed_date: None


In [29]:
assert adapted_item["item_id"] == 5001
assert adapted_item["food_type"] == "cheddar cheese"
assert adapted_item["storage_location"] == "REFRIGERATOR"
assert adapted_item["purchase_date"] == "2026-11-15"
assert adapted_item["prepared_date"] is None
assert adapted_item["food_classification"] == "REGULAR"
assert adapted_item["preparation_state"] is None
assert adapted_item["package_state"] == "OPENED"
assert adapted_item["opened_date"] == "2026-11-24"
assert adapted_item["thawed_date"] is None

print("PASS: Item identifier mapping.")
print("PASS: Food name mapping.")
print("PASS: Storage location preservation.")
print("PASS: Purchase date normalization.")
print("PASS: Missing preparation date preservation.")
print("PASS: Food classification preservation.")
print("PASS: Missing preparation state preservation.")
print("PASS: Package state preservation.")
print("PASS: Opening date normalization.")
print("PASS: Missing thawing date preservation.")

print("\nPantry metadata adaptation: 10/10 checks passed.")

PASS: Item identifier mapping.
PASS: Food name mapping.
PASS: Storage location preservation.
PASS: Purchase date normalization.
PASS: Missing preparation date preservation.
PASS: Food classification preservation.
PASS: Missing preparation state preservation.
PASS: Package state preservation.
PASS: Opening date normalization.
PASS: Missing thawing date preservation.

Pantry metadata adaptation: 10/10 checks passed.


### Backward Compatibility with Legacy Pantry Records

Existing pantry records may not contain the expanded
Phase 5 storage metadata.

The adapter must preserve existing record information
and represent missing metadata as None without
inventing values.

The adapted legacy record is then passed to the
FoodKeeper evaluator to verify that missing metadata
does not result in an unsupported reminder.

This test uses a simulated legacy record and does
not require a live database connection.


In [30]:
legacy_pantry_record = {
    "id": 5002,
    "name": "apple",
    "quantity": 4,
    "unit": "item",
    "storage_location": "REFRIGERATOR",
    "purchase_date": "2026-11-16",
    "expiration_date": None,
}

legacy_adapted_item = storage.adapt_pantry_item(
    legacy_pantry_record
)

print("Adapted legacy record:\n")

for key, value in legacy_adapted_item.items():
    print(f"{key}: {value}")

legacy_result = storage.evaluate_foodkeeper(
    legacy_adapted_item,
    as_of=reference_date,
    foodkeeper_rules=foodkeeper_records,
)

print("\nStorage evaluation:\n")

for key, value in legacy_result.items():
    print(f"{key}: {value}")

Adapted legacy record:

item_id: 5002
food_type: apple
storage_location: REFRIGERATOR
purchase_date: 2026-11-16
prepared_date: None
food_classification: None
preparation_state: None
package_state: None
opened_date: None
thawed_date: None

Storage evaluation:

item_id: 5002
use_priority: UNKNOWN
days_remaining: None
recommended_use_date: None
reminder_status: UNKNOWN
source: None
reason: Food classification is required for this FoodKeeper rule.
storage_range_days: None
storage_range_end_date: None
calculation_basis: None


In [31]:
assert legacy_adapted_item["item_id"] == 5002
assert legacy_adapted_item["food_type"] == "apple"
assert legacy_adapted_item["storage_location"] == "REFRIGERATOR"
assert legacy_adapted_item["purchase_date"] == "2026-11-16"

missing_metadata_fields = [
    "prepared_date",
    "food_classification",
    "preparation_state",
    "package_state",
    "opened_date",
    "thawed_date",
]

for field in missing_metadata_fields:
    assert legacy_adapted_item[field] is None

assert legacy_result["use_priority"] == "UNKNOWN"
assert legacy_result["recommended_use_date"] is None
assert legacy_result["days_remaining"] is None
assert legacy_result["reminder_status"] == "UNKNOWN"
assert legacy_result["source"] is None
assert legacy_result["storage_range_days"] is None
assert legacy_result["storage_range_end_date"] is None
assert legacy_result["calculation_basis"] is None
assert "Food classification is required" in legacy_result["reason"]

print("PASS: Existing record information (4/4 checks).")
print("PASS: Missing metadata preservation (6/6 checks).")
print("PASS: Conservative storage evaluation (9/9 checks).")

print("\nLegacy record compatibility: 19/19 checks passed.")

PASS: Existing record information (4/4 checks).
PASS: Missing metadata preservation (6/6 checks).
PASS: Conservative storage evaluation (9/9 checks).

Legacy record compatibility: 19/19 checks passed.


## 9. Automated Storage Regression Testing

In addition to the functional demonstrations in this
notebook, SmartPantry maintains a dedicated automated
storage guidance test suite.

The following cell executes the existing pytest suite
against the current project implementation.

Test file: tests/test_storage.py

The automated tests cover storage rule loading,
metadata adaptation, calculation eligibility,
reminder calculations, unsupported conditions,
source attribution, and input validation.

No live API calls or database modifications are
required for this test suite.

In [32]:
import subprocess

storage_test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_storage.py",
        "-q",
        "--color=no",
    ],
    cwd=project_root,
    capture_output=True,
    text=True,
)

print(storage_test_run.stdout)

if storage_test_run.stderr:
    print(storage_test_run.stderr)

assert storage_test_run.returncode == 0, (
    "Storage regression tests failed."
)

print("PASS: Automated storage regression testing.")

..................................................................... [ 83%]
..............                                                           [100%]
83 passed, 3 subtests passed in 0.07s

PASS: Automated storage regression testing.


## 10. Phase 5 Results and Conclusions

### Functional Validation Results

| Validation Category | Checks Passed |
|---|---:|
| FoodKeeper dataset structure | 5/5 |
| Purchase-based calculation | 8/8 |
| Opening-based calculation | 8/8 |
| Preparation-based calculation | 8/8 |
| Unsupported conditions | 45/45 |
| OSU informational guidance | 11/11 |
| Pantry metadata adaptation | 10/10 |
| Legacy record compatibility | 19/19 |
| **Total** | **114/114** |

### Automated Regression Results

| Metric | Result |
|---|---:|
| Automated tests passed | 83 |
| Subtests passed | 3 |
| Tests failed | 0 |
| Regression status | PASS |

The notebook assertion checks and automated pytest
results are reported separately because their
validation coverage overlaps.

### Phase 5 Accomplishments

- Expanded storage metadata and database persistence.
- Preserved source-specific storage calculation bases.
- Implemented purchase-based, opening-based, and
  preparation-based reminder calculations.
- Expanded FoodKeeper rule metadata and source traceability.
- Implemented conservative handling of unsupported conditions.
- Preserved OSU guidance as informational content.
- Maintained backward compatibility with legacy pantry records.
- Integrated storage metadata across manual entry, barcode
  recognition, image recognition, and Mochi.
- Improved pantry dashboard storage explanations.
- Expanded the standalone storage recommendation checker.

### Current Limitations

- Calculated reminders are limited to explicitly supported
  food types, storage conditions, and calculation bases.
- FoodKeeper supports calculated reminders for 12 of the
  17 curated records when required conditions are satisfied.
- General refrigeration durations are not automatically
  treated as purchase-based calculations.
- Previously frozen foods require additional validated
  thawing rules before reminders can be calculated.
- OSU records remain informational and are not used
  for calculated reminders.
- Branded product names may require user-assisted mapping
  to supported storage rule names.
- Missing metadata may prevent reminder calculations.
- The prototype does not automatically infer unknown
  storage conditions or dates.

### Conclusion

Phase 5 successfully expanded SmartPantry AI's storage
guidance functionality while maintaining source
traceability, conservative calculation eligibility,
and backward compatibility.

The implementation prioritizes reliable, explainable
recommendations over unsupported predictions.

Use-soon reminders are not expiration dates or food
safety determinations.

When reliable information is unavailable, SmartPantry
returns UNKNOWN rather than inventing a recommendation.

### Supporting Documentation

Detailed implementation and live integration testing:

`tests/results/storage_guidance_refinement_test_results.md`

Automated storage test suite:

`tests/test_storage.py`

Production storage guidance implementation:

`modules/storage_guidance.py`